# 03 · Community Detection — European Air Transport Network

> **Data vintage — June 2014.** A frozen OpenFlights route snapshot, *not* the current network. Every community below describes 2014 connectivity. Community structure is a topological invariant of the route graph, so a dated snapshot is the standard, defensible choice.

**This notebook answers Research Question 2: do natural aviation communities emerge, and do they follow country borders?** It:

1. Runs **Louvain modularity maximisation** on the undirected giant component.
2. Sweeps the resolution parameter (0.5 / 1.0 / 1.5), reporting modularity **Q** and community count — and checks stability across seeds, because Louvain is greedy and stochastic.
3. Writes the chosen partition back to the `community` column of `network.db`.
4. Tests border-alignment three ways — a country-distribution **SQL** query, **NMI/ARI** against the country partition, and Louvain-Q vs the modularity of the national partition.
5. Maps the communities geographically (Plotly) and as a force-directed graph (PyVis).

Graph construction lives in `src/build_graph.py`; paths and the shared palette in `src/utils.py` (imported, not redefined). Only presentation is notebook-specific.

In [ ]:
import sys
import re
import sqlite3
from html import escape as html_escape
import warnings
from pathlib import Path
from contextlib import closing

import numpy as np
import pandas as pd
import networkx as nx
import community as community_louvain          # python-louvain
import plotly.graph_objects as go
import plotly.io as pio
from pyvis.network import Network
from IPython.display import HTML, display
from sklearn.metrics import (
    normalized_mutual_info_score,
    adjusted_rand_score,
)

pio.renderers.default = "notebook"   # renders in static HTML

%load_ext autoreload
%autoreload 2


def _find_src() -> Path:
    """Locate the repo's src/ dir so imports work regardless of launch cwd."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "build_graph.py").exists():
            return p / "src"
    raise FileNotFoundError("Could not locate src/ — run this from inside the repo.")


sys.path.insert(0, str(_find_src()))

import build_graph as bg
import utils

pd.set_option("display.max_columns", None)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Reproducibility / tuning constants (no magic numbers below) ------------ #
SEED: int = 42                       # fixed seed for the reported partition
RESOLUTIONS: tuple[float, ...] = (0.5, 1.0, 1.5)   # resolution sweep
RESOLUTION: float = 1.0              # working resolution chosen after the sweep
N_STAB: int = 20                     # seeds used to gauge sweep stability
WEIGHT_KEY: str = "weight"           # Louvain edge key (see markdown for why weighted)
MIN_LABEL_DEGREE: int = 20           # only label hubs on the PyVis canvas

## 1 · The undirected giant component

Louvain runs on the **undirected** graph: community detection asks *which airports cluster together by their connections*, a symmetric, topological question — so direction (who initiates the route) and the carrier-count weight-as-distance reading that kept NB02's betweenness unweighted are both irrelevant here. We restrict to the **giant component** (the 544-node connected core, 97% of nodes), matching NB02's small-world and eigenvector treatment; the four peripheral fragments carry no meaningful community structure and would only appear as trivial singletons.

In [ ]:
digraph, undirected, edges, airports_eu = bg.build_networks()

# Giant component of the undirected projection (a copy so it is mutable/hashable
# for the community routines).
giant = undirected.subgraph(max(nx.connected_components(undirected), key=len)).copy()
degree = dict(giant.degree())

print(f"Undirected graph : {undirected.number_of_nodes()} nodes, "
      f"{undirected.number_of_edges()} edges")
print(f"Giant component  : {giant.number_of_nodes()} nodes, "
      f"{giant.number_of_edges()} edges "
      f"({giant.number_of_nodes() / undirected.number_of_nodes():.1%} of nodes)")
print(f"Off-giant nodes  : {undirected.number_of_nodes() - giant.number_of_nodes()} "
      f"(will receive community = NULL)")

## 2 · Resolution sweep — how many communities?

**Louvain** greedily maximises **modularity Q**, the fraction of within-community edges minus what a degree-preserving random graph would give; Q > 0 means denser-than-chance internal wiring. The **resolution** γ tunes the scale: higher γ penalises large communities and yields more, smaller ones. Edges enter *weighted* by operating-carrier count — for modularity, weight reads as **coupling strength** (a link served by 8 carriers binds two airports more tightly than one served by 1), which is the correct semantic and, unlike betweenness's distance reading, does not clash with NB02's unweighted choice.

Louvain is greedy and seed-dependent, so a single run can land in a poor local optimum. We report the **seed = 42** partition but also sweep 20 seeds per resolution to expose which γ is *stable* and which merely fragments.

In [ ]:
def louvain_at(res: float, seed: int) -> dict[str, int]:
    """Louvain partition of the giant component at a given resolution and seed."""
    return community_louvain.best_partition(
        giant, resolution=res, random_state=seed, weight=WEIGHT_KEY
    )


sweep_rows = []
for res in RESOLUTIONS:
    # Reported run at the fixed seed.
    part = louvain_at(res, SEED)
    q_seed = community_louvain.modularity(part, giant, weight=WEIGHT_KEY)
    n_seed = len(set(part.values()))

    # Stability across N_STAB seeds.
    counts, qs = [], []
    for s in range(N_STAB):
        p = louvain_at(res, s)
        counts.append(len(set(p.values())))
        qs.append(community_louvain.modularity(p, giant, weight=WEIGHT_KEY))

    sweep_rows.append({
        "resolution": res,
        "communities (seed 42)": n_seed,
        "Q (seed 42)": round(q_seed, 4),
        "Q mean±sd (20 seeds)": f"{np.mean(qs):.4f} ± {np.std(qs):.4f}",
        "community range": f"{min(counts)}–{max(counts)}",
    })

sweep = pd.DataFrame(sweep_rows)
sweep

**Reading the sweep.** γ = 0.5 is **degenerate** — Q collapses (≈ 0.17) and the community count swings wildly across seeds (roughly 15–120): on this dense, small-world graph the greedy optimiser cannot find coherent large communities at low resolution and shatters the network instead. γ = 1.0 is both the **best-scoring and the most stable** setting (6 communities, Q ≈ 0.29, community count pinned at 5–8), and γ = 1.5 subdivides it into ~12 finer, still-stable communities at slightly lower Q. We take **γ = 1.0** as the working partition: highest modularity, reproducible, and coarse enough to read geographically.

### The chosen partition

Louvain community IDs are arbitrary integers that shuffle between seeds, so we **relabel by size** (0 = largest) to make the `community` column and every downstream colour stable. Each community is then tagged with its member airports' country, city and coordinates for the border test and the maps.

In [ ]:
# Final partition at the working resolution, relabelled largest-community-first.
raw_partition = louvain_at(RESOLUTION, SEED)
size_order = pd.Series(raw_partition).value_counts().index
relabel = {old: new for new, old in enumerate(size_order)}
partition = {node: relabel[c] for node, c in raw_partition.items()}

modularity_q = community_louvain.modularity(raw_partition, giant, weight=WEIGHT_KEY)
n_communities = len(set(partition.values()))
print(f"Working partition: {n_communities} communities at γ = {RESOLUTION}, "
      f"Q = {modularity_q:.4f}")

# Per-node frame: community + geography + degree (drives the SQL, maps, PyVis).
nodes = pd.DataFrame([
    {
        "iata": n,
        "city": giant.nodes[n].get("city"),
        "country": giant.nodes[n].get("country"),
        "lat": giant.nodes[n].get("lat"),
        "lon": giant.nodes[n].get("lon"),
        "degree": degree[n],
        "community": partition[n],
    }
    for n in giant.nodes()
])

# Dominant country per community — reused as human-readable map/legend labels.
dominant = (
    nodes.groupby("community")["country"]
    .agg(lambda s: s.value_counts().index[0])
    .to_dict()
)
comm_label = {c: f"{c} · {dominant[c]}" for c in sorted(dominant)}

summary = (
    nodes.groupby("community")
    .agg(airports=("iata", "size"),
         countries=("country", "nunique"),
         dominant_country=("country", lambda s: s.value_counts().index[0]),
         top_countries=("country",
                        lambda s: ", ".join(f"{k}({v})"
                                            for k, v in s.value_counts().head(3).items())))
    .reset_index()
    .sort_values("airports", ascending=False)
)
summary

## 3 · Writing communities to the database

The `community` column was pre-declared NULLable in `schema.sql`, so this is an `UPDATE`, never an `ALTER TABLE` — the same pattern NB02 used for the six centrality columns. Off-giant airports keep `community = NULL` (they were never in the partition), mirroring NB02's off-giant eigenvector = 0.0 decision: a categorical label of *no community* is more honest than forcing them into one.

In [ ]:
UPDATE_COMMUNITY = "UPDATE nodes SET community = :community WHERE iata = :iata"
community_rows = [{"iata": n, "community": int(c)} for n, c in partition.items()]

with closing(sqlite3.connect(utils.DB_PATH)) as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    conn.executemany(UPDATE_COMMUNITY, community_rows)
    conn.commit()
    labelled = pd.read_sql("SELECT COUNT(community) AS n FROM nodes", conn).at[0, "n"]
    unlabelled = pd.read_sql(
        "SELECT COUNT(*) AS n FROM nodes WHERE community IS NULL", conn
    ).at[0, "n"]

off_giant = digraph.number_of_nodes() - giant.number_of_nodes()
print(f"communities written : {labelled} (expect {giant.number_of_nodes()})")
print(f"left NULL (off-giant): {unlabelled} (expect {off_giant})")
assert labelled == giant.number_of_nodes(), "community not written for every giant node"
assert unlabelled == off_giant, "unexpected NULL count"

## 4 · Do communities follow country borders?

Three complementary tests, each stricter than the last:

1. **SQL — dominant country per community.** For each community, which country supplies the most airports, and what share? A high share ⇒ the community *is* essentially one country's domestic network; a low share ⇒ Louvain fused several countries into a super-regional bloc.
2. **NMI / ARI** — information-theoretic and pair-counting agreement between the Louvain partition and the 52-country partition. 1.0 = identical, 0 = independent.
3. **Modularity of the country partition** vs Louvain's Q — if treating each country as a community scores *lower*, then borders are a worse description of the wiring than the regional blocs are.

The query is saved as **`sql/queries/community_country_distribution.sql`** 

In [ ]:
# --- Test 1: dominant-country SQL (read from the saved .sql file) ----------- #
query_path = utils.SQL_QUERIES_DIR / "community_country_distribution.sql"
sql = query_path.read_text(encoding="utf-8")
with closing(sqlite3.connect(utils.DB_PATH)) as conn:
    country_distribution = pd.read_sql(sql, conn)

# --- Test 2: NMI / ARI vs the country partition ----------------------------- #
node_order = list(giant.nodes())
louvain_labels = [partition[n] for n in node_order]
country_labels = [giant.nodes[n].get("country") for n in node_order]
nmi = normalized_mutual_info_score(country_labels, louvain_labels)
ari = adjusted_rand_score(country_labels, louvain_labels)

# --- Test 3: modularity of the country partition vs Louvain ----------------- #
country_ids = {c: i for i, c in enumerate(sorted(set(country_labels)))}
country_partition = {n: country_ids[giant.nodes[n].get("country")] for n in giant.nodes()}
q_country = community_louvain.modularity(country_partition, giant, weight=WEIGHT_KEY)


def intra_edge_fraction(part: dict) -> float:
    """Share of edges whose endpoints share a partition label."""
    intra = sum(1 for u, v in giant.edges() if part[u] == part[v])
    return intra / giant.number_of_edges()


print("Test 1 — dominant country per community:")
print(country_distribution.to_string(index=False))
print(f"\nTest 2 — Louvain vs country partition:  NMI = {nmi:.3f},  ARI = {ari:.3f}")
print("Test 3 — modularity, Louvain vs national borders:")
print(f"   Louvain (6 blocs)      Q = {modularity_q:.4f}   "
      f"(intra-community edges {intra_edge_fraction(partition):.1%})")
print(f"   Country ({len(country_ids)} nations) Q = {q_country:.4f}   "
      f"(intra-country   edges {intra_edge_fraction(country_partition):.1%})")
print(f"   → Louvain modularity is {modularity_q / q_country:.1f}× the national partition's")

### Reading Research Question 2

**Communities emerge, and they are geographic — but they follow *regions*, not borders.** Louvain's six blocs score **Q ≈ 0.29, roughly 2.4× the modularity of the 52-country partition**: about half of all links stay inside a community while only a fifth are domestic, so national boundaries are a markedly worse description of the wiring than the regional blocs are. Agreement with the country partition is only partial (**NMI ≈ 0.47, ARI ≈ 0.24**) — countries explain about half the structure, not all of it.

The blocs split along a clean axis of **internationalisation**. Two are near-mono-national **domestic cores** — a **Norway** bloc (~71% Norwegian, the dense Widerøe/SAS short-haul web up the coast) and a **Turkey** bloc (~78% Turkish, Anatolian domestic plus a Levant fringe): countries whose internal networks are dense enough to form their own module. The rest are **cross-border regional markets** where no single flag dominates — a Franco-Mediterranean bloc binding France, Italy and the Maghreb; a British-Isles/Iberia/holiday bloc; a Nordic-Baltic-plus-Russia bloc; and a Central-Europe/Balkans/Greece bloc shaped as much by German charter and low-cost leisure flows as by adjacency. **These are operating-carrier communities**: NB02's codeshare filter stripped out marketing-alliance structure, so what remains is who actually flies where — geography and low-cost point-to-point networks, not Star/oneworld/SkyTeam branding. (June 2014; today's partition would differ — Wizz/Ryanair bases and 2022 airspace closures post-date the snapshot.)

## 5 · Mapping the communities

Colour each airport by its Louvain community and size it by degree. If communities are geographic, the colours should form contiguous spatial regions rather than scatter at random — the visual companion to the modularity result above.

In [ ]:
fig = go.Figure()
for c in sorted(nodes["community"].unique()):
    sub = nodes[nodes["community"] == c]
    fig.add_trace(go.Scattergeo(
        lon=sub["lon"], lat=sub["lat"], mode="markers",
        name=comm_label[c],
        marker=dict(
            size=3 + np.sqrt(sub["degree"]) * 1.6,
            color=utils.CATEGORICAL_PALETTE[c % len(utils.CATEGORICAL_PALETTE)],
            opacity=0.85,
            line=dict(width=0.3, color=utils.COLORS["bg"]),
        ),
        text=sub["city"] + " (" + sub["iata"] + ") · deg " + sub["degree"].astype(str),
        hoverinfo="text",
    ))

fig.update_geos(
    scope="world", lataxis_range=[33, 73], lonaxis_range=[-27, 47],
    showland=True, landcolor=utils.COLORS["land"],
    showocean=True, oceancolor=utils.COLORS["bg"],
    showcountries=True, countrycolor=utils.COLORS["coastline"],
    coastlinecolor=utils.COLORS["coastline"], bgcolor=utils.COLORS["bg"],
)
fig.update_layout(
    title=dict(text=f"Aviation communities — {utils.FIG_CAPTION}", font=dict(size=15)),
    paper_bgcolor=utils.COLORS["bg"], font_color=utils.COLORS["text"],
    legend=dict(title="Community · dominant country", bgcolor=utils.COLORS["land"],
                bordercolor=utils.COLORS["coastline"], borderwidth=1),
    margin=dict(l=0, r=0, t=40, b=0), height=650,
)

# Static PNG for the README — kaleido may be absent; keep restart-and-run-all clean.
try:
    out = utils.FIGURES_DIR / "community_map.png"
    utils.ensure_dir(utils.FIGURES_DIR)
    fig.write_image(str(out), width=1100, height=750, scale=2)
    print(f"saved {out}")
except Exception as exc:  # noqa: BLE001 — export is optional, display is not
    print(f"PNG export skipped ({type(exc).__name__}); install kaleido to regenerate.")

fig.show()

## 6 · The force-directed view

The map placed each airport at its real coordinates, so it shows *where* communities sit but cannot prove they are real — that clustering is partly geography we imposed. A **force-directed layout** removes geography entirely: node positions come only from the links (connected airports attract, the rest repel). If the Louvain colours still clump here, that is independent confirmation the partition captures genuine dense-connection structure — the visual counterpart to Q being ≈2.4× the country partition. It is also the interactive artefact: drag hubs, hover for names, spot the bridges between blocs. (Honest caveat: at 544 nodes / ~5,200 edges it is a dense hairball, so the geographic map stays the headline figure and this is confirmation + exploration.)

We export **two** self-contained HTML files, both carrying a community legend. A **static** one (`community_network_static.html`, physics off) renders instantly — meant for inline embedding in the Quarto report. The **interactive** one (`community_network.html`) keeps live physics but is tuned to settle in a few seconds rather than hang: a browser simulation of ~5,200 edges only stabilises quickly once the curved-edge support nodes are dropped (`smooth=false`), the iteration count is capped, and the layout is **seeded** from the pre-computed positions so it starts near equilibrium.

In [ ]:
# The two views share ONE seeded layout.
# A shared finaliser strips the dead ../node_modules refs pyvis emits
# and injects a community legend.

CANVAS_SCALE: float = 1400.0
SPRING_K: float = 0.30
LAYOUT_ITERS: int = 100
STABILIZATION_ITERS: int = 250
EMBED_HEIGHT_PX: int = 820

layout = nx.spring_layout(giant, weight=WEIGHT_KEY, seed=SEED, k=SPRING_K, iterations=LAYOUT_ITERS)
xs = np.array([p[0] for p in layout.values()])
ys = np.array([p[1] for p in layout.values()])
pos_px = {
    n: (float(p[0] / max(np.abs(xs).max(), 1e-9) * CANVAS_SCALE),
        float(-p[1] / max(np.abs(ys).max(), 1e-9) * CANVAS_SCALE))   # flip y: screen origin is top-left
    for n, p in layout.items()
}
community_size = nodes["community"].value_counts().to_dict()


def _new_network() -> Network:
    return Network(height="800px", width="100%", bgcolor=utils.COLORS["bg"],
                   font_color=utils.COLORS["text"], notebook=False, cdn_resources="in_line")


def _add_graph(net: Network) -> None:
    """Add every giant-component node (at its pre-computed position) and edge."""
    for n in giant.nodes():
        c, d = partition[n], degree[n]
        x, y = pos_px[n]
        net.add_node(
            n, x=x, y=y,
            label=n if d >= MIN_LABEL_DEGREE else "",       # only hubs are labelled
            title=(f"{giant.nodes[n].get('city')} ({n}) — degree {d}\n"
                   f"community {comm_label[c]}"),
            color=utils.CATEGORICAL_PALETTE[c % len(utils.CATEGORICAL_PALETTE)],
            size=6 + np.sqrt(d) * 2.2, borderWidth=0.5,
        )
    for u, v in giant.edges():
        net.add_edge(u, v, color=utils.COLORS["land"], width=0.4)


def _with_legend(doc: str, subtitle: str) -> str:
    """Strip pyvis's dead ../node_modules refs and inject a title + community legend."""
    doc = re.sub(r'\s*<link[^>]*href="\.\./node_modules/vis/dist/vis\.min\.css"[^>]*>', "", doc)
    doc = re.sub(r'\s*<script[^>]*src="\.\./node_modules/vis/dist/vis\.js"[^>]*>\s*</script>', "", doc)
    swatches = "".join(
        f'<div style="display:flex;align-items:center;margin:2px 0;">'
        f'<span style="width:12px;height:12px;border-radius:50%;'
        f'background:{utils.CATEGORICAL_PALETTE[c % len(utils.CATEGORICAL_PALETTE)]};'
        f'display:inline-block;margin-right:7px;"></span>'
        f'<span>{comm_label[c]} <span style="opacity:.6">({community_size[c]})</span></span></div>'
        for c in sorted(comm_label)
    )
    legend = (
        f'<div style="position:absolute;z-index:999;top:14px;left:14px;'
        f'background:rgba(13,27,42,.85);border:1px solid {utils.COLORS["coastline"]};'
        f'border-radius:8px;padding:10px 13px;color:{utils.COLORS["text"]};'
        f'font-family:system-ui,sans-serif;font-size:12px;max-width:270px;">'
        f'<div style="font-size:13px;font-weight:600;margin-bottom:2px;">'
        f'European aviation communities — {utils.DATA_VINTAGE}</div>'
        f'<div style="opacity:.7;margin-bottom:7px;">{subtitle}</div>'
        f'<div style="font-weight:600;margin-bottom:3px;">Community · dominant country (airports)</div>'
        f'{swatches}'
        f'<div style="opacity:.6;margin-top:7px;">Node size = degree · layout is force-directed '
        f'(connections only, not geography)</div></div>'
    )
    return doc.replace("<body>", "<body>\n" + legend, 1)


# STATIC — rendered inline (NOT saved). Physics off, so it draws instantly. Wrapped in an
# isolated <iframe srcdoc> so the self-contained pyvis document renders without clashing
# with the notebook / Quarto page around it.
static_net = _new_network()
static_net.toggle_physics(False)
_add_graph(static_net)
static_net.set_options('{"edges": {"smooth": false}, "physics": {"enabled": false}, '
                       '"interaction": {"dragNodes": true, "hover": true}}')
static_doc = _with_legend(static_net.generate_html(), "Static layout — rendered inline")
with warnings.catch_warnings():                      # HTML(<iframe>) is intentional here
    warnings.simplefilter("ignore", UserWarning)
    display(HTML(
        f'<iframe srcdoc="{html_escape(static_doc, quote=True)}" width="100%" '
        f'height="{EMBED_HEIGHT_PX}" style="border:none;" '
        f'title="European aviation communities — static"></iframe>'
    ))

# INTERACTIVE — saved to network_viz/. Live physics but tuned to settle in a few seconds:
# smooth off, capped stabilization, seeded from the pre-computed positions.
utils.ensure_dir(utils.NETWORK_VIZ_DIR)
live_net = _new_network()
_add_graph(live_net)
live_net.set_options(
    '{"edges": {"smooth": false},'
    ' "physics": {"enabled": true,'
    '   "barnesHut": {"gravitationalConstant": -8000, "centralGravity": 0.3,'
    '                 "springLength": 120, "springConstant": 0.02, "damping": 0.4},'
    f'   "stabilization": {{"enabled": true, "iterations": {STABILIZATION_ITERS}, "fit": true}},'
    '   "minVelocity": 1.0},'
    ' "interaction": {"dragNodes": true, "hover": true, "hideEdgesOnDrag": true}}'
)
live_path = utils.NETWORK_VIZ_DIR / "community_network.html"
live_net.save_graph(str(live_path))
live_path.write_text(
    _with_legend(live_path.read_text(encoding="utf-8"),
                 "Live physics — give it a few seconds to settle, then drag the hubs"),
    encoding="utf-8",
)

print(f"dynamic : saved {live_path.name}  ({live_path.stat().st_size // 1024} KB, self-contained)")

[↗ Open the interactive force-directed network](../network_viz/community_network.html)

## Summary

**What we built.** A Louvain partition of the 544-airport giant component, chosen by a stability-checked resolution sweep (γ = 1.0, **6 communities, Q ≈ 0.29**), persisted to the `community` column of `network.db`, and tested against national borders three ways — a country-distribution SQL query, NMI/ARI, and a country-partition modularity baseline — plus a geographic map and a force-directed PyVis view.

**What stood out.** **Aviation communities are regional, not national.** They score ~2.4× the modularity of the 52-country partition, half of all links stay within a community versus a fifth within a country, and agreement with borders is only partial (NMI ≈ 0.47). The blocs divide by internationalisation: dense **domestic cores** (Norway ~71%, Turkey ~78%) sit apart from **cross-border regional markets** (Franco-Mediterranean, British-Isles/Iberia, Nordic-Baltic-Russia, Central-Europe/Balkans) shaped by low-cost and charter geography. Because the codeshare filter removed alliance branding, these are *operating-carrier* communities — who really flies where in June 2014.

**Next — `04_resilience_analysis.ipynb`.** The project's centrepiece: percolation under random failure vs targeted hub attack on the giant component, the divergence that is the **Barabási (2000)** result, the critical threshold f_c, and what the loss of Frankfurt, Heathrow or Amsterdam would mean for European connectivity.